# 03 - Modèles BLIP et GIT

Ce notebook est prévu pour Google Colab avec GPU. Il compare le jugement
image-légende de BLIP et GIT sur les mêmes paires du notebook 02, en
précision normale, 8 bits et 4 bits. Les scores et mesures sont sauvegardés
dans Google Drive.

Ce notebook ne doit pas être exécuté dans VS Code.

In [1]:
# Installer les bibliothèques nécessaires dans Google Colab.
!pip -q install datasets transformers accelerate bitsandbytes

In [2]:
# Connecter Google Drive et préparer les dossiers du projet.
from google.colab import drive
from pathlib import Path

drive.mount("/content/drive")
PROJECT_DIR = Path("/content/drive/MyDrive/ia_visuelle")
RESULTS_DIR = PROJECT_DIR / "results"
RESULTS_DIR.mkdir(parents=True, exist_ok=True)


Mounted at /content/drive


In [3]:
import gc
import time

import pandas as pd
import torch
from datasets import load_dataset
from transformers import BitsAndBytesConfig

SEED = 42
N_IMAGES = 500
DEVICE = "cuda" if torch.cuda.is_available() else "cpu"

if DEVICE != "cuda":
    raise RuntimeError("Un GPU Colab est n?cessaire pour ex?cuter ce notebook.")

# V?rifier et r?utiliser le CSV produit par le notebook 01.
references_path = RESULTS_DIR / "flickr8k_references.csv"
if not references_path.exists():
    raise FileNotFoundError(
        f"CSV introuvable : {references_path}. Ex?cutez le notebook 01."
    )
references = pd.read_csv(references_path)
required_columns = ["id"] + [f"caption_{number}" for number in range(5)]
missing = [column for column in required_columns if column not in references]
if missing:
    raise ValueError(f"Colonnes manquantes dans le CSV : {missing}")

# Reprendre exactement les images du notebook 01.
dataset = load_dataset("jxie/flickr8k", split="train")
sample = dataset.shuffle(seed=SEED).select(range(min(N_IMAGES, len(dataset))))
if len(references) != len(sample):
    raise ValueError("Le nombre d images ne correspond pas au notebook 01.")
for number in range(5):
    expected = [example[f"caption_{number}"] for example in sample]
    actual = references[f"caption_{number}"].tolist()
    if actual != expected:
        raise ValueError(f"La s?lection caption_{number} ne correspond pas au notebook 01.")

print(f"Images utilis?es : {len(sample)}")
print(f"GPU utilis? : {torch.cuda.get_device_name(0)}")
# Charger les m?mes paires correctes/incorrectes que dans le notebook 02.
pairs_path = RESULTS_DIR / "baseline_alignment_pairs.csv"
if not pairs_path.exists():
    raise FileNotFoundError(
        f"Paires introuvables : {pairs_path}. Ex?cutez le notebook 02."
    )
pairs_df = pd.read_csv(pairs_path)
required_pair_columns = ["image_id", "caption", "label", "pair_type"]
missing_pairs = [column for column in required_pair_columns if column not in pairs_df]
if missing_pairs:
    raise ValueError(f"Colonnes manquantes dans les paires : {missing_pairs}")
if len(pairs_df) != 2 * len(sample):
    raise ValueError("Le nombre de paires ne correspond pas au notebook 02.")
print(f"Paires image-l?gende ?valu?es : {len(pairs_df)}")


README.md:   0%|          | 0.00/687 [00:00<?, ?B/s]

data/train-00000-of-00002-2f8f6bfa852eac(…): reconstructing file:   0%|          |  0.00B /  414MB            

data/train-00000-of-00002-2f8f6bfa852eac(…): downloading bytes:           |  0.00B            

data/train-00001-of-00002-2173151d8cd6c7(…): reconstructing file:   0%|          |  0.00B /  414MB            

data/train-00001-of-00002-2173151d8cd6c7(…): downloading bytes:           |  0.00B            

data/validation-00000-of-00001-7025a2b59(…): reconstructing file:   0%|          |  0.00B /  138MB            

... (3 outputs left out)

Generating train split:   0%|          | 0/6000 [00:00<?, ? examples/s]

Generating validation split:   0%|          | 0/1000 [00:00<?, ? examples/s]

Generating test split:   0%|          | 0/1000 [00:00<?, ? examples/s]

Images utilis?es : 500
GPU utilis? : Tesla T4
Paires image-l?gende ?valu?es : 1000


## Fonctions communes

Les fonctions suivantes chargent un modèle, calculent le score de chaque
paire image-légende, mesurent le temps total et enregistrent les résultats.
Une configuration
déjà enregistrée est ignorée pour permettre de reprendre après une coupure.

In [4]:
def quantization_config(precision):
    """Créer la configuration bitsandbytes demandée."""
    if precision == "fp16":
        return None
    if precision == "int8":
        return BitsAndBytesConfig(load_in_8bit=True)
    if precision == "int4":
        return BitsAndBytesConfig(
            load_in_4bit=True,
            bnb_4bit_compute_dtype=torch.float16,
        )
    raise ValueError(f"Précision inconnue : {precision}")


def clear_gpu():
    """Libérer le modèle précédent avant de charger le suivant."""
    gc.collect()
    torch.cuda.empty_cache()


def output_path(model_name, precision):
    return RESULTS_DIR / f"judgements_{model_name}_{precision}.csv"


In [5]:
def run_blip(precision):
    """?valuer la coh?rence image-l?gende avec BLIP."""
    from transformers import BlipForConditionalGeneration, BlipProcessor

    model_id = "Salesforce/blip-image-captioning-base"
    processor = BlipProcessor.from_pretrained(model_id)
    model = BlipForConditionalGeneration.from_pretrained(
        model_id,
        quantization_config=quantization_config(precision),
        torch_dtype=torch.float16 if precision == "fp16" else None,
        device_map="auto",
    )
    model.eval()
    torch.cuda.reset_peak_memory_stats()

    rows = []
    start = time.perf_counter()
    for _, pair in pairs_df.iterrows():
        example = sample[int(pair["image_id"])]
        inputs = processor(
            images=example["image"],
            text=pair["caption"],
            return_tensors="pt",
            padding=True,
        )
        inputs = {key: value.to(model.device) for key, value in inputs.items()}
        with torch.no_grad():
            result = model(**inputs, labels=inputs["input_ids"])
        rows.append({
            "image_id": int(pair["image_id"]),
            "caption": pair["caption"],
            "label": int(pair["label"]),
            "pair_type": pair["pair_type"],
            "loss": float(result.loss.detach().cpu()),
        })

    elapsed = time.perf_counter() - start
    peak_memory_mb = torch.cuda.max_memory_allocated() / (1024 ** 2)
    del model, processor
    clear_gpu()
    return rows, elapsed, peak_memory_mb


In [6]:
def run_git(precision):
    """?valuer la coh?rence image-l?gende avec GIT."""
    from transformers import GitForCausalLM, GitProcessor

    model_id = "microsoft/git-base"
    processor = GitProcessor.from_pretrained(model_id)
    model = GitForCausalLM.from_pretrained(
        model_id,
        quantization_config=quantization_config(precision),
        torch_dtype=torch.float16 if precision == "fp16" else None,
        device_map="auto",
    )
    model.eval()
    torch.cuda.reset_peak_memory_stats()

    rows = []
    start = time.perf_counter()
    for _, pair in pairs_df.iterrows():
        example = sample[int(pair["image_id"])]
        inputs = processor(
            images=example["image"],
            text=pair["caption"],
            return_tensors="pt",
            padding=True,
        )
        inputs = {key: value.to(model.device) for key, value in inputs.items()}
        with torch.no_grad():
            result = model(**inputs, labels=inputs["input_ids"])
        rows.append({
            "image_id": int(pair["image_id"]),
            "caption": pair["caption"],
            "label": int(pair["label"]),
            "pair_type": pair["pair_type"],
            "loss": float(result.loss.detach().cpu()),
        })

    elapsed = time.perf_counter() - start
    peak_memory_mb = torch.cuda.max_memory_allocated() / (1024 ** 2)
    del model, processor
    clear_gpu()
    return rows, elapsed, peak_memory_mb


In [ ]:
# Liste des exp?riences ? ex?cuter automatiquement.
# Supprime une ligne pour sauter une exp?rience.
experiments_to_run = [
    ("blip", "fp16"),
    ("blip", "int8"),
    ("blip", "int4"),
    ("git", "fp16"),
    ("git", "int8"),
    ("git", "int4"),
]

experiment_functions = {"blip": run_blip, "git": run_git}
valid_precisions = ["fp16", "int8", "int4"]
summary_path = RESULTS_DIR / "model_judgement_summary.csv"

for model_name, precision in experiments_to_run:
    if model_name not in experiment_functions:
        raise ValueError("Mod?le invalide : " + model_name)
    if precision not in valid_precisions:
        raise ValueError("Pr?cision invalide : " + precision)

    path = output_path(model_name, precision)
    if path.exists():
        print(f"D?j? termin?, ignor? : {path.name}")
        continue

    clear_gpu()
    print(f"D?marrage : {model_name} - {precision}")
    judgements, elapsed, peak_memory_mb = experiment_functions[model_name](precision)
    judgements_df = pd.DataFrame(judgements)
    judgements_df["predicted_label"] = (
        judgements_df.groupby("image_id")["loss"]
        .rank(method="first", ascending=True)
        .eq(1)
        .astype(int)
    )
    accuracy = (judgements_df["predicted_label"] == judgements_df["label"]).mean()
    judgements_df.to_csv(path, index=False)

    row = {
        "model": model_name,
        "precision": precision,
        "accuracy": accuracy,
        "seconds": elapsed,
        "pairs": len(judgements_df),
        "seconds_per_pair": elapsed / len(judgements_df),
        "peak_memory_mb": peak_memory_mb,
    }
    if summary_path.exists():
        summary = pd.read_csv(summary_path)
        summary = summary[~((summary["model"] == model_name) & (summary["precision"] == precision))]
        summary = pd.concat([summary, pd.DataFrame([row])], ignore_index=True)
    else:
        summary = pd.DataFrame([row])
    summary.to_csv(summary_path, index=False)
    print(f"Sauvegard? : {path}")
    print(f"Accuracy : {accuracy:.3f}")
    clear_gpu()


In [8]:
# Afficher le résumé disponible après chaque expérience.
summary_path = RESULTS_DIR / "model_inference_summary.csv"
if summary_path.exists():
    display(pd.read_csv(summary_path))
else:
    print("Aucune expérience n'a encore été enregistrée.")

Aucune expérience n'a encore été enregistrée.


## Vérification

Les fichiers `judgements_blip_*.csv`, `judgements_git_*.csv` et
`model_judgement_summary.csv` seront utilisés par le notebook 04.
Les images et les modèles téléchargés restent dans Colab/Google Drive et
ne doivent pas être ajoutés au dépôt GitHub.